In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

def process_silver_clientes(batch_id: str):
    print("Iniciando processamento da Silver: Clientes")
    
    # 1. Leitura da Bronze (filtrando apenas o lote atual para carga incremental)
    df_bronze = spark.read.table("workspace.default.bronze_clientes") \
                     .filter(F.col("batch_id") == batch_id)
    
    # ==========================================
    # MÓDULO DE QUALIDADE (QUARENTENA)
    # ==========================================
    # Regra: CPF não pode ser nulo
    df_validos = df_bronze.filter(F.col("cpf").isNotNull())
    df_invalidos = df_bronze.filter(F.col("cpf").isNull())
    
    qtd_invalidos = df_invalidos.count()
    if qtd_invalidos > 0:
        print(f"ATENÇÃO: {qtd_invalidos} registros enviados para quarentena (CPF Nulo).")
        df_invalidos.withColumn("motivo_quarentena", F.lit("CPF Nulo")) \
                    .write.format("delta").mode("append") \
                    .saveAsTable("workspace.default.quarentena_clientes")

    # ==========================================
    # PREPARAÇÃO PARA SCD TIPO 2
    # ==========================================
    # Deduplicação dentro do próprio lote (pega sempre a última atualização do cliente no batch)
    window_spec = Window.partitionBy("id_cliente").orderBy(F.col("data_atualizacao").desc())
    df_upsert = df_validos.withColumn("row_num", F.row_number().over(window_spec)) \
                          .filter(F.col("row_num") == 1).drop("row_num")

    # Adiciona colunas de controle do SCD2
    df_upsert = df_upsert.withColumn("is_active", F.lit(True)) \
                         .withColumn("valid_from", F.col("data_atualizacao")) \
                         .withColumn("valid_to", F.lit(None).cast("string"))

    # ==========================================
    # MERGE IDEMPOTENTE (SCD TIPO 2)
    # ==========================================
    tabela_destino = "workspace.default.silver_clientes"
    
    # Verifica se a tabela Silver já existe
    tabela_existe = spark.catalog.tableExists(tabela_destino)
    
    if not tabela_existe:
        # Primeira Carga
        df_upsert.write.format("delta").saveAsTable(tabela_destino)
        print("Tabela silver_clientes criada (Primeira carga).")
    else:
        # Carga Incremental com MERGE
        from delta.tables import DeltaTable
        delta_table = DeltaTable.forName(spark, tabela_destino)
        
        # Lógica: Identifica os registros que mudaram para inativar a versão antiga
        update_cond = """
            target.id_cliente = source.id_cliente AND 
            target.is_active = true AND 
            (target.renda <> source.renda OR target.segmento <> source.segmento)
        """
        
        # Etapa A: Inativa o registro antigo
        delta_table.alias("target").merge(
            df_upsert.alias("source"),
            update_cond
        ).whenMatchedUpdate(set = {
            "is_active": F.lit(False),
            "valid_to": F.col("source.data_atualizacao")
        }).execute()
        
        # Etapa B: Insere o registro novo (ou as inserções puras)
        # O Merge anterior só atualizou. Agora inserimos as linhas novas do dataframe df_upsert
        insert_cond = "target.id_cliente = source.id_cliente AND target.is_active = true"
        
        delta_table.alias("target").merge(
            df_upsert.alias("source"),
            insert_cond
        ).whenNotMatchedInsertAll().execute()
        
        print("MERGE incremental finalizado com sucesso.")

# ==========================================
# EXECUÇÃO
# ==========================================
# Em produção, esse batch_id viria de um orquestrador ou parâmetro de job
# Aqui, vamos buscar o batch_id que acabamos de ingerir na Bronze
ultimo_batch = spark.read.table("workspace.default.bronze_clientes") \
                    .select("batch_id").limit(1).collect()[0][0]

process_silver_clientes(ultimo_batch)